# CLIP TTA v3 — 자동 실행·재개

GPU 런타임을 선택하고 위에서부터 실행하면 됩니다.
B1 → B2 → A → B3 각각 smoke → paper → 보정101/102 → 동결 → 확인201–210 → 평가를 수행합니다.
이미 완료된 계산은 검증 후 건너뛰며 오류가 발생하면 즉시 중단합니다.
전체468만 장의 장시간 작업입니다. 중단 후 같은 STUDY로 다시 실행하세요.
SAR의 FP32 입력/FP16 가중치 오류를 수정한 내장 소스를 사용합니다.
수정 전 결과와 섞지 않도록 새 STUDY를 사용합니다. 기존 Drive 결과는 삭제하지 마세요.


In [ ]:
# 최초 한 번만 설정. 이후 PHASE/SEED를 수동 변경할 필요가 없습니다.
COMBOS = ['B1', 'B2', 'A', 'B3']  # 한 조합만 원하면 ['B1']
GRAD_CKPT = 'auto'
STUDY = 'motivating_exp3_v3_sar_fp16fix'  # 이전 소스의 STUDY와 분리; 재개 시 이 이름 유지
DATA_REVISION = None
PHASE = 'auto'                  # 설치 셀의 GPU 검사에 사용
JOBS = []
for combo in COMBOS:
    JOBS += [(combo, 'smoke', None), (combo, 'paper', 1)]
    JOBS += [(combo, 'calibration', seed) for seed in (101, 102)]
    JOBS += [(combo, 'freeze', None)]
    JOBS += [(combo, 'confirmation', seed) for seed in range(201, 211)]
    JOBS += [(combo, 'evaluate', None)]
print(f'자동 실행: {COMBOS}, 총 {len(JOBS)}개 작업 (완료 작업은 건너뜀)')


In [ ]:
# Drive 연결과 선택적 HF 토큰. 토큰 값은 출력/파일에 저장하지 않습니다.
import os, sys, subprocess
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
assert STUDY and Path(STUDY).name == STUDY and STUDY not in ('.','..')
OUT = Path('/content/drive/MyDrive')/STUDY
REPO = Path('/content')/('motivating_exp3_' + STUDY)
VPY = Path('/content/venv_paper/bin/python')
OUT.mkdir(parents=True,exist_ok=True)
# Compressed dataset cache and temporary files stay on fast local disk.
os.environ['TTA_CACHE_DIR'] = '/content/tta_data_cache'
os.environ['PYTHONUNBUFFERED'] = '1'
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
    if token:
        os.environ['HF_TOKEN'] = token
    del token
except Exception:
    pass


## 소스 복원

처음 실행하면 내장 소스와 이 노트북을 Drive에 저장합니다. 재접속 시 같은 소스만 허용합니다.
다른 버전이 감지되면 덮어쓰지 않고 중단합니다. Drive의 `source_bundle.zip`과
`experiment_notebook.ipynb`를 보관하면 해당 버전으로 돌아올 수 있습니다.

In [ ]:
"""Standard-library-only bootstrap, also embedded verbatim in Colab notebooks."""
import hashlib
import io
from pathlib import Path
import zipfile


def install_bundle(payload, repo, study):
    repo, study = Path(repo), Path(study)
    study.mkdir(parents=True, exist_ok=True)
    digest = hashlib.sha256(payload).hexdigest()
    saved = study / 'source_bundle.zip'
    if saved.exists():
        if hashlib.sha256(saved.read_bytes()).hexdigest() != digest:
            raise ValueError('Drive study has different source code. Open the original saved notebook or choose a NEW study directory.')
    else:
        with saved.open('xb') as f:
            f.write(payload)
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        for member in archive.infolist():
            relative = Path(member.filename)
            if relative.is_absolute() or '..' in relative.parts or member.is_dir():
                raise ValueError('Invalid source bundle path')
            target = repo / relative
            if not target.resolve().is_relative_to(repo.resolve()):
                raise ValueError('Source path escapes repository')
            data = archive.read(member)
            if target.exists():
                if target.read_bytes() != data:
                    raise ValueError(f'Local source was modified: {relative}. Preserve it and select a fresh runtime/source directory.')
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(data)
    return digest


def run_notebook_command(command):
    """Forward Colab stop-cell to the helper so it can stop its scientific job group."""
    import os
    import signal
    import subprocess
    process = subprocess.Popen(list(map(str, command)), start_new_session=True)
    try:
        code = process.wait()
        if code:
            raise subprocess.CalledProcessError(code, command)
    except BaseException:
        if process.poll() is None:
            os.killpg(process.pid, signal.SIGINT)
            try:
                # colab_session.run_logged has its own 10s scientific-group cleanup.
                process.wait(timeout=25)
            except subprocess.TimeoutExpired:
                os.killpg(process.pid, signal.SIGKILL)
                process.wait()
        raise

# Embedded source payload; generated from the current project files.
import base64
PAYLOAD = base64.b64decode('...')
assert hashlib.sha256(PAYLOAD).hexdigest() == '225582965a1a2b18829f60d8467550ae722b05d29b62e359823023865b877755'
BUNDLE_SHA = install_bundle(PAYLOAD, REPO, OUT)
os.chdir(REPO)
print('Source restored:', BUNDLE_SHA, '|', REPO)
# Persist this exact notebook where supported by Colab.
try:
    import json
    from google.colab import _message
    notebook = _message.blocking_request('get_ipynb', timeout_sec=20)['ipynb']
    saved_notebook = OUT/'experiment_notebook.ipynb'
    if not saved_notebook.exists():
        saved_notebook.write_text(json.dumps(notebook,ensure_ascii=False),encoding='utf-8')
except Exception:
    print('Notebook auto-save unavailable; keep your original .ipynb alongside source_bundle.zip.')


## 고정 환경 설치 및 확인

첫 설치에는 수 분 이상 걸릴 수 있습니다. Python 3.11, torch 2.2.1/cu121와 논문 코드를 고정합니다.
해시 검증된 lock으로 설치하고 torchvision 연산·WebP·방법 등록·CUDA 연산을 검사합니다.
설치가 중단되면 이 셀을 다시 실행하세요. 분석 단계는 GPU 없이도 실행할 수 있습니다.

In [ ]:
setup_command = [sys.executable, str(REPO/'paper_repro/setup_colab.py'),
                 '--venv', str(VPY.parent.parent),
                 '--paper-root', str(REPO/'third_party/test-time-adaptation')]
if PHASE in ('freeze','evaluate','status'):
    setup_command.append('--allow-cpu')
if PHASE == 'status':
    print('Status only: dependency installation is unnecessary.')
else:
    run_notebook_command(setup_command)
    # Keep setup evidence on Drive; it contains versions, not credentials.
    import shutil
    for name in ('paper-environment.json','paper-environment.freeze.txt'):
        target = OUT/'environment'/name
        target.parent.mkdir(exist_ok=True)
        shutil.copy2(VPY.parent.parent/name,target)


## 자동 실행 / 재개

아래 셀 하나가 모든 단계와 seed를 순서대로 처리합니다.
중단 후 위에서부터 같은 설정으로 재실행하면 완료 seed를 검증하고 이어갑니다.
Drive의 결과와 source_bundle.zip을 삭제하지 마세요.
실패한 작업 뒤의 실험은 진행하지 않습니다. 완료해도 GPU가 자동 해제되지는 않으므로 런타임을 종료하세요.


In [ ]:
# 노트북 호스트에서 하위 프로세스 출력을 직접 받아 셀에 표시합니다.
import subprocess, signal, json, hashlib

def run_visible(command):
    process = subprocess.Popen(list(map(str, command)), stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1, start_new_session=True)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        code = process.wait()
        if code:
            raise subprocess.CalledProcessError(code, command)
    except BaseException:
        if process.poll() is None:
            os.killpg(process.pid, signal.SIGINT)
            try:
                process.wait(timeout=25)
            except subprocess.TimeoutExpired:
                os.killpg(process.pid, signal.SIGKILL)
                process.wait()
        raise
    finally:
        process.stdout.close()

for index, (combo, phase, seed) in enumerate(JOBS, 1):
    print(f'\n[{index}/{len(JOBS)}] {combo} / {phase} / seed={seed}', flush=True)
    if phase == 'smoke':
        marker = OUT/f'{combo}_notebook_smoke.json'
        if (OUT/f'{combo}_plan.json').exists() or (
            marker.exists() and json.loads(marker.read_text()).get('bundle_sha') == BUNDLE_SHA
            and json.loads(marker.read_text()).get('grad_ckpt') == GRAD_CKPT):
            print('SKIP: 기존 계획 또는 같은 소스/설정의 스모크 기록', flush=True)
            continue
    if phase == 'evaluate' and (OUT/f'{combo}_evaluation.json').exists():
        result = json.loads((OUT/f'{combo}_evaluation.json').read_text())
        digest = hashlib.sha256((OUT/f'{combo}_calibration.json').read_bytes()).hexdigest()
        if (result.get('version') != 3 or result.get('calibration_sha256') != digest
            or {r['seed'] for r in result['runs']} != set(range(201, 211))):
            raise ValueError('기존 평가 파일이 현재 동결/seed 집합과 다릅니다. 덮어쓰지 않습니다.')
        print('SKIP: 동일 동결 임계값으로 완료된 평가', flush=True)
        continue
    command = [sys.executable, str(REPO/'paper_repro/colab_session.py'),
        '--python', str(VPY), '--combo', combo, '--phase', phase,
        '--out', str(OUT), '--grad_ckpt', GRAD_CKPT]
    if seed is not None:
        command += ['--seed', str(seed)]
    if DATA_REVISION is not None:
        command += ['--data_revision', DATA_REVISION]
    run_visible(command)
    if phase == 'smoke':
        (OUT/f'{combo}_notebook_smoke.json').write_text(json.dumps(
            {'bundle_sha': BUNDLE_SHA, 'grad_ckpt': GRAD_CKPT}))
print('선택한 모든 조합의 평가까지 완료했습니다. GPU 런타임을 종료하세요.', flush=True)


## 결과 확인

`<COMBO>_evaluation.json`에 누적 정확도·최악 구간 손해·미감지·지연·실제 오경보가 저장됩니다.
확인 seed는 같은 이미지 풀의 순서 반복이며 독립 배포 데이터셋이 아닙니다.
주 기준과 해석 범위는 내장 `paper_repro/PREREG.md`, 상세 실행 안내는 `paper_repro/COLAB.md`입니다.